In [1]:

# 1. Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import joblib

# Settings
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Load the Dataset

In [2]:
df = pd.read_excel("../data/raw/SiC Data.xlsx")

print("Dataset loaded successfully!")
print("Shape of the dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape of the dataset: (83, 11)

Column names:
['Alloy_code', 'Treatment', 'SiC_vol_perc', 'SiC_size_um', 'av_grain_um', 'YS_MPa', 'UTS_MPa', 'total_perc_elong', 'E_GPa', 'Source', 'DoI']


In [3]:
print("First 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

First 5 rows:


,Alloy_code,Treatment,SiC_vol_perc,SiC_size_um,av_grain_um,YS_MPa,UTS_MPa,total_perc_elong,E_GPa,Source,DoI
0,1,"Homogenized, T6",10.0,10.0,14.00,83.0,176.0,4.10,62.0,Xu et al. (2021),https://doi.org/10.1016/j.jmrt.2021.11.005
1,2,ECAP,10.0,10.0,12.00,246.0,314.0,8.00,65.0,Xu et al. (2021),https://doi.org/10.1016/j.jmrt.2021.11.005
2,3,rolled,10.0,10.0,10.00,284.0,367.0,3.45,83.0,Xu et al. (2021),https://doi.org/10.1016/j.jmrt.2021.11.005
3,4,ECAP+rolled,10.0,10.0,3.00,348.0,411.0,6.30,56.0,Xu et al. (2021),https://doi.org/10.1016/j.jmrt.2021.11.005
4,5,Hot extrusion,2.0,0.5,0.91,480.0,556.0,5.50,87.0,Wu et al. (2020),https://doi.org/10.1080/02670836.2021.1885579



Last 5 rows:


,Alloy_code,Treatment,SiC_vol_perc,SiC_size_um,av_grain_um,YS_MPa,UTS_MPa,total_perc_elong,E_GPa,Source,DoI
78,79,Stir casting,0.5,0.2,16.8,162.0,232.6,7.15,60.77,Deng et al. (2010),https://doi.org/10.1016/j.jallcom.2010.05.159
79,80,Stir casting,1.0,0.2,16.5,164.0,234.7,7.11,61.34,Deng et al. (2010),https://doi.org/10.1016/j.jallcom.2010.05.159
80,81,Stir casting,1.5,0.2,16.0,167.0,238.0,7.10,61.96,Deng et al. (2010),https://doi.org/10.1016/j.jallcom.2010.05.159
81,82,Stir casting,2.0,0.2,16.0,169.5,240.0,7.04,62.54,Deng et al. (2010),https://doi.org/10.1016/j.jallcom.2010.05.159
82,83,Stir casting,3.0,0.2,15.3,174.7,245.9,6.99,63.71,Deng et al. (2010),https://doi.org/10.1016/j.jallcom.2010.05.159


## Basic Information & Missing Values

In [4]:
print("Data Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())

Data Types:
Alloy_code            int64
Treatment               str
SiC_vol_perc        float64
SiC_size_um         float64
av_grain_um         float64
YS_MPa              float64
UTS_MPa             float64
total_perc_elong    float64
E_GPa               float64
Source                  str
DoI                     str
dtype: object

Missing Values:
Alloy_code          0
Treatment           0
SiC_vol_perc        0
SiC_size_um         0
av_grain_um         0
YS_MPa              0
UTS_MPa             0
total_perc_elong    0
E_GPa               0
Source              0
DoI                 0
dtype: int64

Statistical Summary:


,Alloy_code,SiC_vol_perc,SiC_size_um,av_grain_um,YS_MPa,UTS_MPa,total_perc_elong,E_GPa
count,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000
mean,42.000000,8.174699,23.942169,9.287640,170.069084,245.681928,5.662909,65.910274
std,24.103942,7.593496,29.141028,21.514954,96.872694,108.509058,3.956358,20.907683
min,1.000000,0.500000,0.200000,0.740000,28.300000,95.100000,0.500000,40.224532
25%,21.500000,2.000000,5.000000,1.399001,108.510274,181.500000,2.480000,54.800000
50%,42.000000,5.000000,15.000000,2.100000,140.000000,221.000000,4.870000,61.774490
75%,62.500000,13.000000,30.000000,8.025000,205.500000,279.150000,7.070000,67.000000
max,83.000000,30.000000,105.000000,118.000000,515.000000,622.000000,18.300000,159.600000


### Data Cleaning & Feature Selection

In [5]:
# Keep only useful columns
df_clean = df[['Treatment', 'SiC_vol_perc', 'SiC_size_um', 'av_grain_um', 'UTS_MPa']].copy()

print("Selected columns:")
print(df_clean.columns.tolist())
print("\nShape after selection:", df_clean.shape)
display(df_clean.head())

Selected columns:
['Treatment', 'SiC_vol_perc', 'SiC_size_um', 'av_grain_um', 'UTS_MPa']

Shape after selection: (83, 5)


,Treatment,SiC_vol_perc,SiC_size_um,av_grain_um,UTS_MPa
0,"Homogenized, T6",10.0,10.0,14.00,176.0
1,ECAP,10.0,10.0,12.00,314.0
2,rolled,10.0,10.0,10.00,367.0
3,ECAP+rolled,10.0,10.0,3.00,411.0
4,Hot extrusion,2.0,0.5,0.91,556.0


### Encode Categorical Column (Treatment)

In [6]:
# Encode the Treatment column
le = LabelEncoder()
df_clean['Treatment_encoded'] = le.fit_transform(df_clean['Treatment'])

# Show the mapping
print("Treatment Encoding Mapping:")
for original, encoded in zip(le.classes_, range(len(le.classes_))):
    print(f"{original} → {encoded}")

print("\nData after encoding:")
display(df_clean.head())

Treatment Encoding Mapping:
CCDFed → 0
Centrifugal casting → 1
ECAP → 2
ECAP+rolled → 3
Homogenized, T6 → 4
Homogenized, T7 → 5
Homogenized, T8 → 6
Homogenized, T9 → 7
Hot extrusion → 8
Semi-solid stirring → 9
Simulated → 10
Stir casting → 11
rolled → 12

Data after encoding:


,Treatment,SiC_vol_perc,SiC_size_um,av_grain_um,UTS_MPa,Treatment_encoded
0,"Homogenized, T6",10.0,10.0,14.00,176.0,4
1,ECAP,10.0,10.0,12.00,314.0,2
2,rolled,10.0,10.0,10.00,367.0,12
3,ECAP+rolled,10.0,10.0,3.00,411.0,3
4,Hot extrusion,2.0,0.5,0.91,556.0,8
